In [37]:
import csv
import os
from typing import Annotated ,TypedDict
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition

In [46]:
os.environ["GOOGLE_API_KEY"] = "Enter API Key here"
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    timeout=60,
    
)


In [39]:
@tool
def check_pantry(ingredient: str) -> str:
    """Checks if a specific ingredient is in stock by reading pantry_inventory.csv."""
    with open("pantry_inventory.csv", mode="r", newline="") as file:
        reader = csv.DictReader(file)
        for row in reader:
            if row["ingredient"].strip().lower() == ingredient.strip().lower():
                return row["status"]
    return "Ingredient not found in pantry log."
tools = [check_pantry]
llm_with_tools = llm.bind_tools(tools)


In [40]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


In [41]:
def chef_agent(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(AgentState)
builder.add_node("chef", chef_agent)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chef")
builder.add_conditional_edges("chef", tools_condition)
builder.add_edge("tools", "chef")

agent = builder.compile()


In [45]:
result = agent.invoke({"messages": [("yes")]})
print(result["messages"][-1].text)


It looks like you said "yes," but I'm not sure what you're responding to! 

Could you please let me know what you'd like to do? For example, we could check your pantry for a specific ingredient, or look for a recipe.


In [43]:
# Create a real CSV inventory file in your current folder on D:
csv_content = """ingredient,status
paneer,In stock (5 kg fresh)
cheese,Out of stock!
bun,In stock (24 brioche buns)
tomato,In stock (2 kg ripe)
"""

with open("pantry_inventory.csv", "w") as file:
    file.write(csv_content)

print("pantry_inventory.csv created successfully!")


pantry_inventory.csv created successfully!
